Primo test - RNN con Attention

1. Preparazione ambiente (importazione librerie e controllo GPU)

In [ ]:
import pandas as pd
import numpy as np
import re
import torch

print(f"PyTorch Version: {torch.__version__}")
#imposto la GPU come device principale (per usarlo sulla mia macchina, se non è disponibile userà la CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Sto usando: {device}")

2. Analisi e preprocessing dei dati (caricamento del file berg.txt e pulizia del testo)

In [ ]:
def preprocess_sentence(w):
    #trasforma tutto in minuscolo e toglie spazi all'inizio e alla fine
    w = str(w).lower().strip()

    #stacco la punteggiatura dalle parole per considerarla come token separati
    # Es: "ciao." diventa "ciao ."
    w = re.sub(r"([?.!,¿])", r" \1 ", w)
    w = re.sub(r'[" "]+', " ", w)

    #tengo le lettere, i numeri, la punteggiatura e gli accenti tipici dei dialetti/italiano
    #includo quindi à, è, é, ì, í, ò, ó, ù, ú, ö, ü
    w = re.sub(r"[^a-zàèéìíòóùúöü0-9?.!,¿']+", " ", w)
    
    w = w.strip()
    
    #inserisco i token per far capire alla rete quando inizia e finisce la frase
    w = '<sos> ' + w + ' <eos>'
    return w

#test per vedere il risultato
test_phrase = "L'è svèelt com ü fölmen." # Dal tuo file berg.txt
print("Prima:", test_phrase)
print("Dopo: ", preprocess_sentence(test_phrase))

3. Architettura del vocabolario (tokenizzazione e indicizzazione)

In [ ]:
#leggo il file ignorando la terza colonna (fonti)
df = pd.read_csv("berg.txt", sep="\t", header=None, usecols=[0, 1], names=["Bergamasco", "Italiano"])

#pulizia di eventuali righe vuote
df = df.dropna()

#preprocessing di entrambe le colonne
df['Bergamasco_pulito'] = df['Bergamasco'].apply(preprocess_sentence)
df['Italiano_pulito'] = df['Italiano'].apply(preprocess_sentence)

#risultato delle prime righe
display(df.head())

In [ ]:
class Vocabulary:
    def __init__(self):
        #inizializzazione dei dizionari con i 4 token di base
        self.word2index = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
        self.index2word = {0: "<pad>", 1: "<sos>", 2: "<eos>", 3: "<unk>"}
        self.n_words = 4  #contatore delle parole (parte da 4 perché ci sono già i token)

    def add_sentence(self, sentence):
        #frase divisa in singole parole (sugli spazi)
        for word in sentence.split(' '):
            self.add_word(word)

    def add_word(self, word):
        #se la parola non è mai stata vista, viene inserita e le viene assegnato un ID
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.index2word[self.n_words] = word
            self.n_words += 1

#si creano i due vocabolari vuoti per le due lingue
vocab_berg = Vocabulary()
vocab_ita = Vocabulary()

#si leggono riga per riga i dataframe e si popolano i vocabolari
for frase in df['Bergamasco_pulito']:
    vocab_berg.add_sentence(frase)

for frase in df['Italiano_pulito']:
    vocab_ita.add_sentence(frase)

#stampa il numero di parole dei vocabolari
print(f"Grandezza vocabolario Bergamasco: {vocab_berg.n_words} parole uniche")
print(f"Grandezza vocabolario Italiano: {vocab_ita.n_words} parole uniche")

#test cercando l'ID di una parola (visiù)
parola_test = "visiù"
if parola_test in vocab_berg.word2index:
    print(f"L'ID della parola '{parola_test}' è: {vocab_berg.word2index[parola_test]}")

4. Pipeline di input per PyTorch (dataset e DataLoader)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

#1. come PyTorch deve leggere e trasformare le frasi
class DialectDataset(Dataset):
    def __init__(self, df, vocab_berg, vocab_ita):
        self.df = df
        self.vocab_berg = vocab_berg
        self.vocab_ita = vocab_ita

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Prendiamo le frasi pulite
        frase_berg = self.df.iloc[idx]['Bergamasco_pulito']
        frase_ita = self.df.iloc[idx]['Italiano_pulito']

        #ogni parola è convertita nel suo ID. Se una parola non c'è viene assegnato 3 (<unk>)
        tensor_berg = [self.vocab_berg.word2index.get(w, 3) for w in frase_berg.split(' ')]
        tensor_ita = [self.vocab_ita.word2index.get(w, 3) for w in frase_ita.split(' ')]

        #tensori di numeri (ID) per entrambe le frasi
        return torch.tensor(tensor_ita), torch.tensor(tensor_berg)

#2. funzione per pareggiare le lunghezze con gli zeri
def pad_collate(batch):
    berg_batch, ita_batch = zip(*batch)
    
    #applico gli zeri (padding_value=0) alla fine delle sequenze più corte
    berg_padded = pad_sequence(berg_batch, batch_first=True, padding_value=0)
    ita_padded = pad_sequence(ita_batch, batch_first=True, padding_value=0)
    
    return berg_padded, ita_padded

#3. creazione oggetto Dataset
dataset = DialectDataset(df, vocab_berg, vocab_ita)

#4. creazione il DataLoader (porta i dati alla GPU). 
#batch_size=2 per testare ilfile berg.txt limitato nella quantità
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=pad_collate)

#test per vedere come appaiono i batch di dati dopo il padding
for berg_b, ita_b in dataloader:
    print("BATCH BERGAMASCO (Tensore di Numeri):")
    print(berg_b)
    print(f"Dimensioni: {berg_b.shape} (2 frasi, lunghezza massima {berg_b.shape[1]})")
    
    print("\nBATCH ITALIANO (Tensore di Numeri):")
    print(ita_b)
    print(f"Dimensioni: {ita_b.shape} (2 frasi, lunghezza massima {ita_b.shape[1]})")
    break #fermo dopo aver visto solo il primo batch

5. Definizione del modello (RNN con attention)

In [ ]:
import torch.nn as nn

class RnnEncoder(nn.Module):
    def __init__(self, src_vocab, embedding_dim, hidden_units):
        super(RnnEncoder, self).__init__()
        #uso .n_words come definito nella nostra classe Vocabulary
        self.src_vocab = src_vocab 
        vocab_size = src_vocab.n_words 

        #inizializza il layer di embedding
        self.embedding_layer = nn.Embedding(vocab_size, embedding_dim)

        #inizializza la GRU (batch_first=False)
        self.single_gru = nn.GRU(input_size=embedding_dim, hidden_size=hidden_units, num_layers=1, batch_first=False)

    def forward(self, x):
        #dato che il dataloader crea tensori [batch_size, max_len]
        #li trasponiamo in [max_len, batch_size] per far funzionare la GRU
        x = x.transpose(0, 1) 

        #passo x nell'embedding e poi nella RRN
        embedded = self.embedding_layer(x)
        output, hidden_state = self.single_gru(embedded)

        return output, hidden_state

In [ ]:
import torch.nn.functional as F

class RnnDecoder(nn.Module):
    def __init__(self, trg_vocab, embedding_dim, hidden_units):
        super(RnnDecoder, self).__init__()
        # Adattamento per la nostra classe Vocabulary
        self.trg_vocab = trg_vocab
        vocab_size = trg_vocab.n_words

        #inizializza il layer di embedding
        self.embedding_layer = nn.Embedding(vocab_size, embedding_dim)

        #inizializza i layer per calcolare lo score di attention
        self.w_1 = nn.Linear(hidden_units, hidden_units)
        self.w_2 = nn.Linear(hidden_units, hidden_units)
        self.v_a = nn.Linear(hidden_units, 1)

        #decoder della GRU
        self.single_gru = nn.GRU(input_size=embedding_dim+hidden_units, hidden_size=hidden_units, num_layers=1, batch_first=True)

        #layer di output
        self.fully_connected_layer = nn.Linear(hidden_units, vocab_size)


    def compute_attention(self, dec_hs, enc_output):
        dec_hs_perm = dec_hs.permute(1,0,2)
        enc_output_perm = enc_output.permute(1,0,2)
        score = self.v_a(torch.tanh(self.w_1(enc_output_perm) + self.w_2(dec_hs_perm)))

        attention_weights = F.softmax(score, dim=1)
        context_vector = torch.sum(attention_weights*enc_output_perm, dim=1)

        return context_vector, attention_weights

    def forward(self, x, dec_hs, enc_output):
        context_vector, attention_weights = self.compute_attention(dec_hs, enc_output)
        embedding_vectors = self.embedding_layer(x)
        concatenate = torch.cat((embedding_vectors, context_vector.unsqueeze(1)), dim=2)
        
        output, dec_hs = self.single_gru(concatenate, dec_hs)
        fc_out = self.fully_connected_layer(output.squeeze(1))

        return fc_out, dec_hs, attention_weights

    def decode_step(self, inputs, enc_output, dec_hs):
        assert inputs.shape[0] == enc_output.shape[1] == dec_hs.shape[1], 'batch_size must be the same across tensors'
        fc_out, dec_hs, attention_weights = self(inputs[:,-1].unsqueeze(1), dec_hs, enc_output)
        return fc_out, dec_hs

6. Training e evoluzione della loss

In [ ]:
import torch.optim as optim
import time

#la funzione serve per dire alla rete quanto sta sbagliando
#ignora il token <pad> (ID 0) così non viene penalizzata per gli zeri di padding
def loss_function(real, pred):
    mask = real.ge(1).float() 
    loss_ = F.cross_entropy(pred, real, reduction='none') * mask
    return torch.mean(loss_)

def train_translation_model(encoder, decoder, dataloader, vocab_ita, epochs=10):
    #gli ottimizzatori aggiornano i pesi per farli migliorare
    encoder_optimizer = optim.Adam(encoder.parameters(), lr=0.001)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=0.001)
    
    print("Inizio Addestramento...")
    
    for epoch in range(epochs):
        start_time = time.time()
        total_loss = 0
        
        encoder.train()
        decoder.train()
        
        #prende un blocco di frasi alla volta
        for batch_idx, (src, trg) in enumerate(dataloader):
            #sposto i dati sulla scheda video
            src, trg = src.to(device), trg.to(device)
            batch_size = src.size(0)
            
            encoder_optimizer.zero_grad()
            decoder_optimizer.zero_grad()
            loss = 0
            
            #1. encoder legge il bergamasco: frasi -> "Riassunto Mentale"
            enc_output, dec_hidden = encoder(src)
            
            #2. inizia a generare l'italiano
            dec_input = torch.tensor([[1]] * batch_size, device=device) 
            
            #3. il decoder genera una parola alla volta
            for t in range(1, trg.size(1)):
                #il decoder fa le sue previsioni guardando la parola precedente e il riassunto dell'encoder
                predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
                
                #si calcola quanto la parola prevista sia lontana da quella reale che doveva indovinare
                loss += loss_function(trg[:, t], predictions)
                
                #invece di dargli in pasto la sua previsione (probabilmente sbagliata) 
                #si forza dandogli in pasto la parola vera come prossimo input, per aiutarlo a imparare più in fretta
                dec_input = trg[:, t].unsqueeze(1)
                
            batch_loss = loss / int(trg.size(1))
            total_loss += batch_loss.item()
            
            #aggiornamento dei pesi: apprendimento
            batch_loss.backward()
            encoder_optimizer.step()
            decoder_optimizer.step()
            
        print(f'Epoca {epoch+1:02d}/{epochs} | Errore (Loss): {total_loss/len(dataloader):.4f} | Tempo: {time.time()-start_time:.1f}s')

EMBEDDING_DIM = 256
HIDDEN_UNITS = 256

#ora l'encoder usa il vocabolario italiano per leggere
encoder_model = RnnEncoder(vocab_ita, EMBEDDING_DIM, HIDDEN_UNITS).to(device)

#mentre il decoder usa il vocabolario Bergamasco per generare
decoder_model = RnnDecoder(vocab_berg, EMBEDDING_DIM, HIDDEN_UNITS).to(device)

#avvia l'addestramento dicendo alla funzione che il target è vocab_berg
train_translation_model(encoder_model, decoder_model, dataloader, vocab_berg, epochs=50)

7. Test di inferenza e risultati

In [22]:
def translate_ita_to_berg(sentence, encoder, decoder, vocab_src, vocab_trg, max_length=50):
    encoder.eval()
    decoder.eval()

    sentence_pulita = preprocess_sentence(sentence)
    
    #prendo il vocabolario italiano per leggere l'input
    tokens = [vocab_src.word2index.get(w, 3) for w in sentence_pulita.split(' ')]
    src_tensor = torch.tensor(tokens).unsqueeze(0).to(device) 

    with torch.no_grad():
        enc_output, dec_hidden = encoder(src_tensor)
        dec_input = torch.tensor([[1]], device=device) 
        translated_words = []

        for _ in range(max_length):
            predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
            predicted_id = predictions.argmax(1).item()

            if predicted_id == 2:
                break

            #uso il vocabolario Bergamasco per scrivere l'output
            translated_word = vocab_trg.index2word.get(predicted_id, "<unk>")
            translated_words.append(translated_word)
            dec_input = torch.tensor([[predicted_id]], device=device)

    return " ".join(translated_words)

#provo un test con una frase non presente nel dataset
frase_test = "è veloce come un gatto al sole"

traduzione = translate_ita_to_berg(frase_test, encoder_model, decoder_model, vocab_ita, vocab_berg)

print(f"ITALIANO: {frase_test}")
print(f"PREDETTO IN BERGAMASCO: {traduzione}")


ITALIANO: è veloce come un gatto al sole
PREDETTO IN BERGAMASCO: l'è ü bradipo .
